# 03 · Analyze

**Phase 4 deliverables:** emotions on the full clean dataset, stage and theme tags, summary tables, hypothesis tests, five charts, and a 30-post hand check.

Data rules: overall shares and brand comparisons use the **random** sample only. Within-stage comparisons use **random + stage boost**. Shares are "% of posts expressing the emotion" (a post can express several). Cells with fewer than 30 posts are flagged, not hidden.

*Independent student project, not affiliated with WHOOP.*

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import analysis, charts, emotions, tagging

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)
clean = pd.read_csv(analysis.CLEAN)
print(f"{len(clean):,} clean rows")

## 1 · Emotion model on all clean rows

Saves progress every 256 rows to `data/clean/emotion_scores_full.csv`. If it stops, just run the cell again and it picks up where it left off. Expect roughly 5–15 minutes on your Mac.

In [ ]:
clf = emotions.load_model()
scores = analysis.score_full(clf, clean)

## 2 · Tag stages and themes

In [ ]:
df = analysis.build(clean, scores)
df.to_csv(ROOT / "data" / "clean" / "analysis_full.csv", index=False)   # has text: stays local
tables = analysis.summaries(df)
print(f"{len(df):,} rows analyzed")
print("\nStage-tagged posts (random + boost):")
tables["stage_counts"]

In [ ]:
cover = pd.DataFrame({
    "stage tagged": (df["stage"] != "untagged").groupby(df["brand"]).mean(),
    "any theme": (df["themes"] != "").groupby(df["brand"]).mean(),
    "any core emotion": (df["primary_emotion"] != "none").groupby(df["brand"]).mean(),
    "worry words": df["worry_words"].groupby(df["brand"]).mean(),
})
cover.round(3)

## 3 · Overall emotion mix by brand (random sample)

In [ ]:
analysis.wide(tables["overall_by_brand"], "brand").round(3)

## 4 · Emotion by stage

In [ ]:
print("WHOOP:")
display(analysis.wide(tables["stage_by_brand"].query("brand == 'WHOOP'"), "stage").round(3))
print("All four brands, each brand × stage:")
analysis.wide(tables["stage_by_brand"], ["brand", "stage"]).round(3)

## 5 · Hypothesis tests

Two-proportion z-tests. "Supported" means the difference is in the predicted direction, p < 0.05, and both groups have at least 30 posts. These are associations in a self-selected sample, not causes. Report the numbers either way: a hypothesis that isn't supported is still a finding.

In [ ]:
tests = analysis.hypothesis_tests(df)
tests.round(3)

In [ ]:
print("Cancelling posts that give price/cost as a reason (money words only, not membership/renew), by brand:")
tables["price_in_cancelling"].round(3)

## 6 · Charts

In [ ]:
made = charts.make_all(tables)
for name, (path, caption) in made.items():
    display(Markdown(f"**{name}** — {caption}"))
    display(Image(filename=str(path), width=900))

## 7 · Hand-check 30 posts

20 random posts (5 per brand) plus 10 that matched the worry word list. For each row, fill in:
- `emotions_ok`: **y** if the listed emotions are reasonable, **n** if not
- `stage_ok`: **y** if the stage tag is right (or correctly "untagged"), **n** if not

When you export from Numbers, use **File → Export To → CSV** and **uncheck "Include table names"**.

In [ ]:
sheet = analysis.handcheck_sheet(df)
sheet_path = ROOT / "data" / "clean" / "handcheck_30.csv"
sheet.to_csv(sheet_path, index=False)
print(f"Saved {sheet_path.relative_to(ROOT)}")
sheet[["check_type", "brand", "stage", "emotions", "text_model"]]

In [ ]:
# Run after filling in the CSV
first = sheet_path.read_text().splitlines()[0]
done = pd.read_csv(sheet_path, skiprows=0 if first.startswith("check_type") else 1)
yes = lambda s: s.astype(str).str.strip().str.lower().str.startswith("y")
for kind, g in done.groupby("check_type"):
    print(f"{kind:12} emotions ok: {yes(g['emotions_ok']).sum()}/{len(g)}   stage ok: {yes(g['stage_ok']).sum()}/{len(g)}")
print(f"{'all':12} emotions ok: {yes(done['emotions_ok']).sum()}/{len(done)} "
      f"({yes(done['emotions_ok']).mean():.0%}),  error rate {1 - yes(done['emotions_ok']).mean():.0%}")

### 7b · Fresh check of the worry word list

The word list was tightened after the first hand check (negations like "not so worried" and "without getting obsessed" are now ignored). Those fixes were made using the same 30 posts, so check **10 new** worry matches to get an honest hit rate. Fill in `emotions_ok` only (y = the post really expresses worry or anxiety).

In [ ]:
fresh = analysis.handcheck_sheet(df, n_random=0, n_worry=10, seed=99)
fresh = fresh[~fresh["text_model"].isin(sheet["text_model"])]
fresh_path = ROOT / "data" / "clean" / "handcheck_worry_v2.csv"
fresh.to_csv(fresh_path, index=False)
print(f"Saved {len(fresh)} fresh worry matches to {fresh_path.relative_to(ROOT)}")
fresh[["brand", "emotions", "text_model"]]

In [ ]:
# Run after filling in the CSV
first = fresh_path.read_text().splitlines()[0]
done2 = pd.read_csv(fresh_path, skiprows=0 if first.startswith("check_type") else 1)
ok2 = done2["emotions_ok"].astype(str).str.strip().str.lower().str.startswith("y")
print(f"Worry word list, fresh check: {ok2.sum()} of {len(done2)} matches are real worry ({ok2.mean():.0%})")

### 7c · Check the cancelling tag

The first check found only 14 of 30 cancelling tags were real ("switched to the wrist", "cancel the order", advice like "cancel your subscription and…"). The rule now needs first person, an account object or a competitor. This sheet is a **fresh** sample (posts from the first check are excluded): 20 WHOOP and 10 Garmin cancelling posts, with the phrase that triggered the tag. Fill in `really_leaving`: **y** if the writer has left, is leaving or is seriously thinking about leaving, **n** otherwise.

In [ ]:
def trigger(t):
    m = tagging.STAGE_RES["cancelling"].search(t)
    return m.group(0) if m else ""
cx = df[df["stage"] == "cancelling"]
first_check = ROOT / "data" / "clean" / "handcheck_cancelling.csv"
if first_check.exists():  # keep the fresh check fresh
    cx = cx[~cx["text_model"].isin(pd.read_csv(first_check)["text_model"])]
cancel_check = pd.concat([
    cx[cx["brand"] == "WHOOP"].sample(n=min(20, (cx["brand"] == "WHOOP").sum()), random_state=17),
    cx[cx["brand"] == "Garmin"].sample(n=min(10, (cx["brand"] == "Garmin").sum()), random_state=17),
])[["brand", "type", "sample", "text_model"]]
cancel_check.insert(3, "trigger", cancel_check["text_model"].map(trigger))
cancel_check = cancel_check.assign(really_leaving="").reset_index(drop=True)
cancel_path = ROOT / "data" / "clean" / "handcheck_cancelling_v2.csv"
cancel_check.to_csv(cancel_path, index=False)
print(f"Saved {cancel_path.relative_to(ROOT)}")
print("Most common triggers among ALL cancelling posts:")
print(cx["text_model"].map(trigger).str.lower().value_counts().head(10))

## 8 · Save the summary tables (aggregate only: safe for GitHub)

In [ ]:
path = analysis.save_tables(tables, tests)
print(f"Saved tables to {path.relative_to(ROOT)}:", sorted(p.name for p in path.glob('*.csv')))

## Phase 4 done when

- [ ] Full run complete and every chart has its caption
- [ ] Hypothesis table reviewed
- [ ] 30 posts hand-checked, error rate written down

Send Claude: the stage counts, the coverage table, the overall-by-brand table, the WHOOP stage table, the hypothesis tests, the price-in-cancelling table, the five chart PNGs and your hand-check result.